# medrag - Demo Colab

Pipeline RAG medical complet : PDF -> Markdown -> Hierarchy -> Chunks -> ChromaDB -> Recherche hybride

**Pre-requis** : un PDF medical + votre cle API Modal (optionnel)

## 1. Installation

On fixe d'abord les versions compatibles de torch/torchvision, puis on installe medrag.

In [ ]:
# Etape 1 : Fixer torch/torchvision compatible (evite le bug torchvision::nms)
!pip install -q torch torchvision --upgrade

# Etape 2 : Installer medrag depuis GitHub
!pip install -q git+https://github.com/Sadoubar/Med_Rag.git

# Verification
import medrag
print(f'medrag version : {medrag.__version__}')

## 2. Configuration

Renseignez votre cle API Modal et l'URL de votre service d'extraction.

In [ ]:
# === CONFIGURATION (A MODIFIER) ===

API_URL = 'https://sadoubar--api.modal.run'   # URL de votre API Modal
API_KEY = 'VOTRE_CLE_ICI'                      # Remplacer par votre sk_main_...

# Ou bien utiliser un secret Colab :
# from google.colab import userdata
# API_KEY = userdata.get('MODAL_API_KEY')

print(f'API URL : {API_URL}')
print(f'API Key : {API_KEY[:12]}...' if len(API_KEY) > 12 else 'NON CONFIGUREE')

## 3. Upload du PDF

Uploadez votre PDF medical via le file picker Colab.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
size_mb = os.path.getsize(PDF_PATH) / 1024**2
print(f'\nPDF : {PDF_PATH} ({size_mb:.1f} MB)')

## 4. Pipeline complet : Extract + Hierarchy + Chunk + Index

Une seule commande fait tout le travail.

In [ ]:
from medrag import MedRAG

rag = MedRAG(
    chroma_path='./chromadb',
    collection_name='medrag_demo',
    extract_api_url=API_URL,
    extract_api_key=API_KEY,
)

print('Pipeline en cours : extract -> hierarchy -> chunk -> index')
print('(premiere execution : telechargement du modele ~500 MB)\n')

result = rag.add_pdf(PDF_PATH)

print(f'\nSource         : {result["source"]}')
print(f'Chunks crees   : {result["chunks_created"]}')
print(f'Total indexes  : {result["total_indexed"]}')

## 5. Recherche hybride

BM25 (lexical) + Dense (semantique) + fusion RRF.

In [ ]:
query = 'traitement paludisme enfant'

results = rag.search(query, top_k=5)

print(f'Question : {query}')
print(f'Resultats : {len(results)}\n')
print('=' * 70)

for i, r in enumerate(results, 1):
    section = ' > '.join(r.section_path)
    sim = f'{r.cosine_sim:.3f}' if r.cosine_sim else 'N/A'
    print(f'\n#{i} [{r.source.upper()}] RRF={r.score:.4f} | Cosine={sim}')
    print(f'   Section : {section}')
    print(f'   Apercu  : {r.text[:200]}...')
    print('-' * 70)

## 6. Tester vos propres questions

In [ ]:
# Modifiez la question et relancez cette cellule
mes_questions = [
    'posologie amoxicilline enfant',
    'signes anemie grossesse',
    'diagnostic tuberculose',
]

for q in mes_questions:
    results = rag.search(q, top_k=3)
    print(f'\nQ: {q}')
    for i, r in enumerate(results, 1):
        section = ' > '.join(r.section_path[-2:])  # 2 derniers niveaux
        print(f'  #{i} [{r.source}] {section} (RRF={r.score:.4f})')
    print()

## 7. Ajouter un 2eme PDF au corpus (optionnel)

Le mode `--no-reset` est automatique : chaque `add_pdf()` ajoute au corpus existant.

In [ ]:
# Decommentez pour ajouter un 2eme PDF

# uploaded2 = files.upload()
# PDF_PATH_2 = list(uploaded2.keys())[0]
# result2 = rag.add_pdf(PDF_PATH_2)
# print(f'Chunks ajoutes : {result2["chunks_created"]}')
# print(f'Total corpus   : {rag.count()}')

## 8. Benchmark (optionnel)

In [ ]:
queries_benchmark = [
    'traitement paludisme grave',
    'posologie amoxicilline',
    'anemie ferriprive grossesse',
    'fievre enfant moins 5 ans',
    'deshydratation nourrisson',
    'tuberculose multiresistante',
]

report = rag.benchmark(queries_benchmark, top_k=4)

print(f'Requetes testees      : {report["total_queries"]}')
print(f'Avec source "both"    : {report["queries_with_both"]}')
print(f'Resultats/requete moy : {report["avg_results_per_query"]:.1f}')
print(f'Distribution sources  : {report["source_distribution"]}')

## 9. Utilisation etape par etape (mode avance)

Si vous voulez controler chaque etape individuellement.

In [ ]:
from medrag.extract import extract_pdf
from medrag.hierarchy import diagnose_text, reconstruct_text
from medrag.chunk import chunk_markdown_text

# 1. Extraction
# markdown = extract_pdf('guide.pdf', api_url=API_URL, api_key=API_KEY)

# 2. Diagnostic hierarchie
# diag = diagnose_text(markdown)
# print(f'Diagnostic : {diag.diagnosis}')
# print(f'H1={diag.h1} H2={diag.h2} H3={diag.h3} H4={diag.h4}')

# 3. Reconstruction
# reconstructed = reconstruct_text(markdown)

# 4. Chunking
# chunks, stats = chunk_markdown_text(reconstructed, 'mon_guide')
# print(f'Chunks : {stats.total_chunks}')
# print(f'Filtres : {stats.filtered_meta_sections} meta, {stats.filtered_orphan_chunks} orphelins')

print('Decommentez les lignes ci-dessus pour tester etape par etape')